# YOLOv8 Segmentation Training
COCO Polygon Annotations를 사용한 인스턴스 세그멘테이션 학습

In [17]:
import os
import json
import yaml
from pathlib import Path
from datetime import datetime
import glob
from sklearn.model_selection import train_test_split
import numpy as np
from ultralytics import YOLO
import cv2
import logging

# 작업 디렉토리 설정
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir) if 'notebooks' in notebook_dir else notebook_dir
os.chdir(project_root)

print(f"프로젝트 루트: {os.getcwd()}")
print("✓ 필요한 라이브러리 로드 완료")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

프로젝트 루트: /data/th/KETI/Grounded-SAM-2
✓ 필요한 라이브러리 로드 완료


In [18]:
# COCO annotation 파일 찾기
def find_annotation_files(annotations_dir="annotations"):
    all_files = glob.glob(os.path.join(annotations_dir, "*_coco.json"))
    annotation_files = sorted([f for f in all_files if 'merged' not in os.path.basename(f).lower() and 'backup' not in os.path.basename(f).lower()])
    logging.info(f"찾은 annotation 파일: {len(annotation_files)}개")
    return annotation_files

annotation_files = find_annotation_files()
print(f"총 {len(annotation_files)}개의 annotation 파일 발견")

2025-11-21 16:24:46,278 - INFO - 찾은 annotation 파일: 143개


총 143개의 annotation 파일 발견


In [19]:
# 여러 COCO annotation 파일 병합
def merge_coco_annotations(annotation_files):
    merged_data = {
        "info": {"description": "Merged COCO Annotations", "version": "1.0", "year": datetime.now().year},
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }
    
    next_image_id = 1
    next_annotation_id = 1
    next_category_id = 1
    
    for file_path in annotation_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 카테고리 병합
            file_category_map = {}
            for category in data.get("categories", []):
                cat_name = category.get("name")
                old_cat_id = category.get("id")
                
                existing_cat = next((c for c in merged_data["categories"] if c.get("name") == cat_name), None)
                if existing_cat:
                    file_category_map[old_cat_id] = existing_cat["id"]
                else:
                    new_cat_id = next_category_id
                    next_category_id += 1
                    file_category_map[old_cat_id] = new_cat_id
                    merged_data["categories"].append({"id": new_cat_id, "name": cat_name, "supercategory": "thing"})
            
            # 이미지 및 annotation 병합
            file_image_map = {}
            for image in data.get("images", []):
                old_image_id = image.get("id")
                new_image_id = next_image_id
                next_image_id += 1
                file_image_map[old_image_id] = new_image_id
                
                new_image = image.copy()
                new_image["id"] = new_image_id
                merged_data["images"].append(new_image)
            
            for annotation in data.get("annotations", []):
                new_annotation = annotation.copy()
                new_annotation["id"] = next_annotation_id
                next_annotation_id += 1
                new_annotation["image_id"] = file_image_map.get(annotation.get("image_id"))
                new_annotation["category_id"] = file_category_map.get(annotation.get("category_id"))
                merged_data["annotations"].append(new_annotation)
        
        except Exception as e:
            logging.error(f"파일 병합 실패 {file_path}: {e}")
            continue
    
    logging.info(f"병합 완료: {len(merged_data['images'])}개 이미지, {len(merged_data['annotations'])}개 annotation")
    return merged_data

merged_coco_data = merge_coco_annotations(annotation_files)
print(f"✓ 병합 완료: {len(merged_coco_data['images'])}개 이미지, {len(merged_coco_data['categories'])}개 카테고리")

2025-11-21 16:24:46,469 - INFO - 병합 완료: 143개 이미지, 429개 annotation


✓ 병합 완료: 143개 이미지, 3개 카테고리


In [20]:
# COCO Polygon을 YOLO Segmentation 형식으로 변환
def coco_to_yolo_seg(merged_coco_data, output_images_dir="dataset_seg/images", output_labels_dir="dataset_seg/labels", images_dir="images"):
    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)
    
    image_map = {img['id']: img for img in merged_coco_data.get('images', [])}
    used_images = []
    
    for image_id, img_info in image_map.items():
        img_filename = img_info['file_name']
        img_path = os.path.join(images_dir, img_filename)
        
        if not os.path.exists(img_path):
            logging.warning(f"이미지를 찾을 수 없음: {img_filename}")
            continue
        
        # 심볼릭 링크 생성
        link_path = os.path.join(output_images_dir, img_filename)
        if not os.path.exists(link_path):
            os.symlink(os.path.abspath(img_path), link_path)
        
        img_width, img_height = img_info['width'], img_info['height']
        
        # 라벨 파일 생성
        image_annotations = [ann for ann in merged_coco_data.get('annotations', [])
                           if ann.get('image_id') == image_id]
        
        label_file_path = os.path.join(output_labels_dir, f"{img_filename.split('.')[0]}.txt")
        with open(label_file_path, 'w') as label_file:
            for ann in image_annotations:
                if 'segmentation' in ann and len(ann['segmentation']) > 0:
                    class_id = ann['category_id'] - 1
                    
                    # Polygon을 정규화된 좌표로 변환
                    for polygon in ann['segmentation']:
                        if len(polygon) >= 6:  # 최소 3개의 점 필요
                            normalized_coords = []
                            for i in range(0, len(polygon), 2):
                                if i + 1 < len(polygon):
                                    x_norm = polygon[i] / img_width
                                    y_norm = polygon[i + 1] / img_height
                                    normalized_coords.extend([x_norm, y_norm])
                            
                            if len(normalized_coords) >= 6:
                                label_file.write(f"{class_id} {' '.join(map(str, normalized_coords))}\n")
        
        used_images.append(img_filename)
    
    logging.info(f"YOLO Segmentation 변환 완료: {len(used_images)}개 이미지")
    return used_images, merged_coco_data.get('categories', [])

used_images, categories = coco_to_yolo_seg(merged_coco_data)
print(f"✓ YOLO Segmentation 형식 변환 완료: {len(used_images)}개 이미지")

2025-11-21 16:24:47,014 - INFO - YOLO Segmentation 변환 완료: 143개 이미지


✓ YOLO Segmentation 형식 변환 완료: 143개 이미지


In [21]:
# Train/Val 셋 분리 (80:20)
train_images, val_images = train_test_split(used_images, test_size=0.2, random_state=42)

print(f"✓ Train/Val 분리 완료")
print(f"  - Train: {len(train_images)}개 이미지")
print(f"  - Val: {len(val_images)}개 이미지")

# Train/Val 리스트를 텍스트 파일로 저장
os.makedirs("dataset_seg", exist_ok=True)

with open("dataset_seg/train.txt", 'w') as f:
    for img in train_images:
        f.write(os.path.abspath(os.path.join("dataset_seg/images", img)) + "\n")

with open("dataset_seg/val.txt", 'w') as f:
    for img in val_images:
        f.write(os.path.abspath(os.path.join("dataset_seg/images", img)) + "\n")

print(f"✓ Train/Val 리스트 파일 생성 완료")

✓ Train/Val 분리 완료
  - Train: 114개 이미지
  - Val: 29개 이미지
✓ Train/Val 리스트 파일 생성 완료


In [22]:
# data.yaml 생성
class_names = {cat['id'] - 1: cat['name'] for cat in categories}

data_yaml = {
    'path': os.path.abspath("dataset_seg"),
    'train': os.path.abspath("dataset_seg/train.txt"),
    'val': os.path.abspath("dataset_seg/val.txt"),
    'nc': len(categories),
    'names': class_names
}

yaml_path = "dataset_seg/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✓ data.yaml 생성 완료")
print(f"\n클래스 정보:")
for class_id, class_name in sorted(class_names.items()):
    print(f"  {class_id}: {class_name}")

✓ data.yaml 생성 완료

클래스 정보:
  0: DOOR
  1: JOG
  2: MEMORY


In [23]:
# YOLOv8 Segmentation 모델 학습
job_id = datetime.now().strftime('%Y%m%d_%H%M%S')

try:
    logging.info("YOLOv8-Seg 모델 초기화 및 학습 시작")
    model = YOLO('yolov8s-seg.pt')
    
    results = model.train(
        data=yaml_path,
        epochs=10,
        batch=16,
        imgsz=640,
        project="trained_models",
        name=f"yolo_seg_{job_id}",
        verbose=False
    )
    
    print(f"✓ Segmentation 모델 학습 완료")
    
except Exception as e:
    print(f"✗ 학습 실패: {e}")
    import traceback
    traceback.print_exc()

2025-11-21 16:24:47,045 - INFO - YOLOv8-Seg 모델 초기화 및 학습 시작


New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.203 🚀 Python-3.11.0 torch-2.3.1+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_seg/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo_seg_20251121_162447, n

In [24]:
# Val 셋 검증
try:
    logging.info("Val 셋 검증 시작")
    val_results = model.val()
    
    print(f"✓ Val 셋 검증 완료")
    print(f"\n세그멘테이션 성능 메트릭:")
    print(f"  - mAP@50 (Mask): {val_results.seg.map50:.4f}")
    print(f"  - mAP@50-95 (Mask): {val_results.seg.map:.4f}")
    print(f"  - Precision: {val_results.seg.mp:.4f}")
    print(f"  - Recall: {val_results.seg.mr:.4f}")
    
except Exception as e:
    print(f"✗ 검증 실패: {e}")

2025-11-21 16:25:20,561 - INFO - Val 셋 검증 시작


Ultralytics 8.3.203 🚀 Python-3.11.0 torch-2.3.1+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,780,761 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1172.2±219.7 MB/s, size: 436.2 KB)
val: Scanning /data/th/KETI/Grounded-SAM-2/dataset_seg/labels.cache... 29 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29/29 40.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s1.7s
                   all         29         87      0.967      0.855      0.982      0.769      0.879      0.766      0.822      0.307
Speed: 1.6ms preprocess, 11.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /data/th/KETI/Grounded-SAM-2/trained_models/yolo_seg_20251121_1624472
✓ Val 셋 검증 완료

세그멘테이션 성능 메트릭:
  - mAP@50 (Mask): 0.8223
  - mAP@50-95 (Mask): 0.3068
  - Precision: 0.8788
  